
# YOLOv8-Pose Training for SFP Port Detection
> **Project:** AIC SFP Assembly  
> **Target:** Detecting 4 corners of SFP Ports  
> **Instructions:** Follow the steps below. Ensure your Runtime Type is set to **T4 GPU** (Runtime -> Change runtime type).

### Step 1: Install Environment and check GPU availability

In [ ]:
# 1. Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found — go to Runtime → Change runtime type → T4 GPU")

# 2. Install Dependencies
!pip install ultralytics -q

### Step 2: Upload Dataset

In [ ]:
from google.colab import files
import zipfile, os
print("Please upload the '{name}_colab.zip' generated by your local script.")
uploaded = files.upload()  # select your dataset.zip

# Extract the dataset
zip_name = list(uploaded.keys())[0]
!mkdir -p /content/dataset
!unzip -q {zip_name} -d /content/dataset
print("Dataset extracted to /content/dataset")

### Step 3: Create Training Configuration

In [ ]:
import os

# Define dataset configuration
# Note: In Step 2, we extracted to /content/dataset, so paths must match.
config_content = """
path: /content/dataset
train: train/images
val: val/images

kpt_shape: [4, 3] # 4 corners (x, y, visibility)

names:
  0: sfp_port_0
  1: sfp_port_1
"""

with open('/content/dataset_config.yaml', 'w') as f:
    f.write(config_content)

print("Configuration 'dataset_config.yaml' created.")

### Step 4: Training

#### Step 4.1: Configure Training

In [ ]:
# This will be the name of the folder where best.pt is saved
EXPERIMENT_NAME = "sfp_port_v1"

# Project name (the main folder)
PROJECT_NAME = "sfp_assembly_project"

# Training settings
EPOCHS = 50           # Number of full passes through the dataset
IMAGE_SIZE = 640      # Resolution (YOLO standard is 640)
BATCH_SIZE = 16       # Number of images processed at once (T4 GPU handles 16 well)
PATIENCE = 10         # Stop training if no improvement after 10 epochs

# Model selection
# Options: yolov8n-pose.pt (Fastest), yolov8s-pose.pt (More accurate, slower)
BASE_MODEL = "yolov8n-pose.pt"

print(f"Configuration set for experiment: {EXPERIMENT_NAME}")

#### Step 4.2: Start Training

In [ ]:
from ultralytics import YOLO

# Load the base model
model = YOLO(BASE_MODEL)

# Start training
# The results will be saved in: {PROJECT_NAME}/{EXPERIMENT_NAME}/
results = model.train(
    data='/content/dataset_config.yaml',
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    device=0,         # Use GPU
    exist_ok=True     # Overwrite if folder exists
)

print(f"mAP50:     {metrics.pose.map50:.3f}")
print(f"mAP50-95:  {metrics.pose.map:.3f}")
print(f"Training complete! Results saved in {PROJECT_NAME}/{EXPERIMENT_NAME}")

### Step 5: Download the model

In [ ]:
from google.colab import files
files.download('/content/runs/pose/sfp_assembly_project/sfp_gpu_run/weights/best.pt')